### Implementação de Fuzzy C-means utilizando GloVe

In [201]:
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import plotly.express as px
import skfuzzy as fuzz
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score


In [203]:
def load_glove(file_path):
    embeddings = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

def words_to_vectors(words, embeddings, dimension=50):
    vectors = []
    for word in words:
        vector = embeddings.get(word)
        if vector is not None:
            vectors.append(vector)
        else:
            print(f"'{word}' not found in GloVe vocabulary. Using zero vector.")
            vectors.append(np.zeros(dimension))
    return np.array(vectors)

glove_file_path = './GloVe/glove.6B.50d.txt'

glove_embeddings = load_glove(glove_file_path)


In [ ]:
words = ['harvard', 'learning', 'intelligence']

word_vectors = words_to_vectors(words, glove_embeddings)
print(word_vectors)

[[-8.5970e-01  1.1120e+00 -2.9970e-01 -1.1093e+00  1.5653e-01 -1.3244e-01
  -1.0520e+00 -9.2620e-01 -5.2920e-01 -2.4501e-01 -2.2653e-01  2.5299e-01
  -9.9125e-02 -4.0640e-01  9.7853e-04 -3.5808e-02 -1.8689e-01  7.1157e-01
  -4.4480e-01  8.6651e-01  5.4339e-01  5.9826e-01 -3.1584e-02 -4.6351e-01
  -8.5038e-02 -1.8902e+00  1.1140e-01 -7.5604e-01 -1.6965e+00 -3.9752e-01
   1.2976e+00 -3.4127e-01 -2.2890e-01 -1.4524e+00 -2.9855e-01 -2.0297e-01
  -4.4211e-01  1.1521e+00  1.5059e+00 -4.8819e-01 -2.1176e-01 -3.6186e-01
  -9.1108e-02  9.5266e-01  2.0254e-01  1.0068e-01  6.9316e-01  2.6215e-01
  -9.0986e-01  5.9507e-01]
 [ 2.0461e-01  4.8659e-01 -5.5308e-01 -2.7019e-01  2.6336e-01  1.5751e-01
  -2.8994e-01 -5.1824e-01  5.1829e-02  3.6225e-01  3.7077e-01  1.3220e-01
  -6.1377e-02 -5.3606e-01 -3.4733e-01 -4.3981e-02 -8.6744e-02  7.8305e-01
   4.1422e-01  2.7996e-02  2.3433e-01  9.8844e-01 -4.1049e-01  6.2060e-01
   1.3966e+00 -6.5427e-01 -1.8221e-01 -1.0293e+00 -1.4741e-02 -2.5384e-01
   3.2270e+

In [ ]:
wordsim_path_file = './WordSim_353/wordsim_relatedness_goldstandard.txt'
df = pd.read_csv(wordsim_path_file, sep='\t', header=None)
print(df.head())
words = pd.concat([df[0], df[1]]).str.lower().drop_duplicates()
word_vectors = words_to_vectors(words, glove_embeddings)
print('#words = ', len(words))

           0          1     2
0   computer   keyboard  7.62
1  Jerusalem     Israel  8.46
2     planet     galaxy  8.11
3     canyon  landscape  7.53
4       OPEC    country  5.63
#words =  346


In [ ]:
# perplexity = np.arange(10, 300, 10)
# divergence = []

# for i in perplexity:
#     model = TSNE(n_components=2, init="pca", perplexity=i)
#     reduced = model.fit_transform(word_vectors)
#     divergence.append(model.kl_divergence_)
# fig = px.line(x=perplexity, y=divergence, markers=True)
# fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="Divergence")
# fig.update_traces(line_color="red", line_width=1)
# fig.show()

In [ ]:
tsne = TSNE(n_components=2,perplexity=20, init='pca', random_state=0)
word_vectors_tsne = tsne.fit_transform(word_vectors)

tsne.kl_divergence_

1.2116248607635498

In [ ]:
fig = px.scatter(x=word_vectors_tsne[:, 0], y=word_vectors_tsne[:, 1], text=words)
fig.update_layout(
    title="t-SNE visualization of WordSim_353 dataset",
    xaxis_title="First t-SNE",
    yaxis_title="Second t-SNE",
    width=800,
    height=500
)
fig.show()

In [ ]:
n_clusters = 8
fuzz_par = 1.1

fcm_model = fuzz.cluster.cmeans(word_vectors.T, c=n_clusters, m=fuzz_par, error=0.05, maxiter=1000, init=None)
cluster_membership = np.argmax(fcm_model[1], axis=0)

df_membership = pd.DataFrame(fcm_model[1].T, index=words, columns=[f"Cluster {i}" for i in range(n_clusters)])

top_words_per_cluster = {}

top_n = 100

for i in range(n_clusters):
    # Seleciona as top_n palavras ordenadas pelo grau de pertencimento ao cluster i
    top_entries = df_membership[f"Cluster {i}"].sort_values(ascending=False).head(top_n)
    top_words_per_cluster[f"Cluster {i}"] = [f"{word} ({value:.4f})" for word, value in zip(top_entries.index, top_entries.values)]

df_top_words = pd.DataFrame.from_dict(top_words_per_cluster, orient="index")

display(df_top_words)

df_membership = pd.DataFrame(fcm_model[1].T, index=words, columns=[f"Cluster {i}" for i in range(n_clusters)])

df_membership = df_membership.map(lambda x: f"{x:.4f}")

display(df_membership)

fcm_im = fcm_model[1] @ fcm_model[1].T

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
Cluster 0,shore (0.9973),area (0.9912),canyon (0.9899),sea (0.9862),mars (0.9779),proximity (0.9682),coast (0.9624),forest (0.9620),observation (0.9580),graveyard (0.9537),...,stroke (0.0752),development (0.0749),library (0.0734),mouth (0.0732),arrangement (0.0583),opera (0.0580),string (0.0572),board (0.0526),possession (0.0512),property (0.0500)
Cluster 1,round (1.0000),match (0.9999),team (0.9996),season (0.9994),game (0.9989),cup (0.9976),record (0.9973),competition (0.9970),victory (0.9966),football (0.9941),...,hill (0.0146),board (0.0145),senate (0.0142),journal (0.0140),confidence (0.0137),flight (0.0134),war (0.0132),attempt (0.0130),combination (0.0130),king (0.0129)
Cluster 2,government (0.9992),plan (0.9969),possibility (0.9946),withdrawal (0.9940),issue (0.9935),delay (0.9917),israel (0.9904),ministry (0.9850),planning (0.9846),effort (0.9843),...,holy (0.0684),car (0.0603),street (0.0599),string (0.0550),admission (0.0550),weather (0.0536),tiger (0.0511),ticket (0.0509),accommodation (0.0500),environment (0.0492)
Cluster 3,computer (0.9980),hardware (0.9974),software (0.9966),internet (0.9966),phone (0.9920),video (0.9917),network (0.9900),media (0.9791),information (0.9768),telephone (0.9681),...,century (0.0599),clinic (0.0592),property (0.0569),virtuoso (0.0547),center (0.0544),opera (0.0513),exhibit (0.0509),industry (0.0507),preservation (0.0480),preparation (0.0437)
Cluster 4,lover (0.9992),brother (0.9977),man (0.9975),doctor (0.9972),mother (0.9969),girl (0.9890),love (0.9878),monk (0.9860),life (0.9841),magician (0.9757),...,boxing (0.0677),keyboard (0.0674),television (0.0655),rooster (0.0631),physics (0.0618),chemistry (0.0609),hundred (0.0588),closet (0.0588),governor (0.0574),category (0.0557)
Cluster 5,popcorn (0.9995),drink (0.9991),coffee (0.9983),egg (0.9967),sugar (0.9955),cabbage (0.9952),shower (0.9949),seafood (0.9920),liquid (0.9864),cucumber (0.9854),...,hundred (0.0169),wealth (0.0165),galaxy (0.0153),graveyard (0.0152),word (0.0149),music (0.0148),depression (0.0147),reservation (0.0143),movie (0.0140),smart (0.0138)
Cluster 6,price (1.0000),stock (1.0000),market (1.0000),profit (0.9998),trading (0.9996),dollar (0.9995),interest (0.9995),investor (0.9988),credit (0.9985),currency (0.9982),...,governor (0.0069),depression (0.0064),journal (0.0064),senate (0.0062),admission (0.0062),string (0.0058),criterion (0.0056),territory (0.0054),fertility (0.0054),project (0.0053)
Cluster 7,nature (0.9940),morality (0.9900),isolation (0.9826),cognition (0.9814),psychology (0.9772),prejudice (0.9757),gender (0.9751),experience (0.9631),disability (0.9607),importance (0.9583),...,marriage (0.1568),drought (0.1564),scientist (0.1431),word (0.1324),recommendation (0.1311),exhibit (0.1194),maradona (0.1165),century (0.1097),proton (0.1008),critic (0.1007)


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7
computer,0.0002,0.0001,0.0001,0.9980,0.0005,0.0001,0.0004,0.0006
jerusalem,0.2352,0.0042,0.6499,0.0125,0.0542,0.0029,0.0028,0.0383
planet,0.9396,0.0018,0.0018,0.0148,0.0204,0.0086,0.0005,0.0124
canyon,0.9899,0.0006,0.0007,0.0018,0.0024,0.0029,0.0003,0.0014
opec,0.0132,0.0124,0.2438,0.0101,0.0061,0.0618,0.6328,0.0197
...,...,...,...,...,...,...,...,...
voyage,0.9257,0.0097,0.0076,0.0045,0.0401,0.0045,0.0006,0.0072
string,0.0572,0.4427,0.0550,0.1509,0.1608,0.0313,0.0058,0.0963
smile,0.0329,0.0128,0.0059,0.0260,0.4231,0.4390,0.0015,0.0588
cucumber,0.0056,0.0019,0.0005,0.0017,0.0029,0.9854,0.0006,0.0014


In [ ]:
fig = px.scatter(x=word_vectors_tsne[:, 0], y=word_vectors_tsne[:, 1], text=words, color=cluster_membership)
fig.update_layout(
    title="t-SNE visualization of WordSim_353 dataset with FCM",
    xaxis_title="First t-SNE",
    yaxis_title="Second t-SNE",
    width=1500,
    height=900
)
fig.show()

fcm_im = fcm_model[1].T @ fcm_model[1]

df_membership.index = np.array(df_membership.index)
df_membership['cluster'] = cluster_membership

df_membership_ord = df_membership.sort_values(by='cluster')
df_membership_ord = df_membership_ord.drop(columns='cluster')
# display(df_membership_ord)

df_membership_ord = np.array(df_membership_ord).astype(float)
fcm_im_ord = df_membership_ord @ df_membership_ord.T

# Create a subplot with two columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Affinity matrix", "Ordered affinity matrix"),
    column_widths=[0.5, 0.5]
)

# Add the first heatmap
fig.add_trace(
    px.imshow(fcm_im, color_continuous_scale='plasma').data[0],
    row=1, col=1
)

# Add the second heatmap
fig.add_trace(
    px.imshow(fcm_im_ord, color_continuous_scale='plasma').data[0],
    row=1, col=2
)

# Update layout
fig.update_layout(
    title="Comparison of FCM Membership Matrices",
    width=1200,
    height=600
)

fig.show()

In [ ]:
# Encontre as palavras que tem maior valor de pertinência menor que 0.5
words_below_threshold = []
for i in range(words.shape[0]):
    words_below_threshold.append(float(df_membership.iloc[i,cluster_membership[i]]) < 0.4)

# Exiba as palavras que têm maior valor de pertinência menor que 0.5
display(df_membership[words_below_threshold])

,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,cluster
fbi,0.0757,0.0048,0.3663,0.3196,0.1731,0.0030,0.0042,0.0532,2
drug,0.0258,0.0286,0.1597,0.2125,0.1646,0.0552,0.0669,0.2869,7
concert,0.1898,0.2515,0.0237,0.1356,0.3405,0.0369,0.0027,0.0193,4
luxury,0.3038,0.0237,0.0230,0.2329,0.0423,0.1907,0.1498,0.0339,0
weapon,0.0780,0.0284,0.1171,0.3902,0.1163,0.0377,0.0040,0.2284,3
decoration,0.2038,0.0195,0.0176,0.0997,0.1110,0.1585,0.0045,0.3855,7
arrangement,0.0583,0.0051,0.2281,0.3833,0.0377,0.0132,0.0221,0.2522,3
size,0.2027,0.0206,0.0107,0.1790,0.0154,0.3500,0.1236,0.0979,5
development,0.0749,0.0075,0.3467,0.1134,0.0053,0.0009,0.0621,0.3893,7
population,0.3833,0.0181,0.1304,0.0201,0.0494,0.0101,0.0517,0.3369,0


### Critérios de avaliação de Clusters

In [204]:
# gerando duas gaussianas centradas em (0,0) e (5,5) com variância 1 e rotuladas com 0 e 1
Xy1 = np.random.normal(0, 1, (100, 2))
Xy1 = np.concatenate((Xy1, np.zeros((100, 1))), axis=1)

Xy2 = np.random.normal(5, 1, (100, 2))
Xy2 = np.concatenate((Xy2, np.ones((100, 1))), axis=1)

Xy = np.concatenate((Xy1, Xy2), axis=0)

cE = np.mean([Xy[j,:-1] for j in range(Xy.shape[0])], axis=0)

# plotando os dados
fig = px.scatter(x=Xy[:, 0], y=Xy[:, 1], color=['blue' if label == 0 else 'red' for label in Xy[:, 2]])

# Add a marker for the center point
fig.add_scatter(x=[cE[0]], y=[cE[1]], mode='markers+text', text=['Centroid global'], textposition='top center', marker=dict(color='green', size=10))

fig.update_layout(
    title="Gaussian Mixture",
    xaxis_title="X1",
    yaxis_title="X2",
    width=800,
    height=500
)
fig.show()

#### Internal Clustering Indices
- Avalia a qualidade de uma clusterização sem usar rótulos/infomrações externas;
- Baseia-se somente na estrutura intrínseca dos dados;

- **Silhouette Coefficient**: mede a similaridade de um ponto com o seu próprio _cluster_ em comparação com outros _clusters_. O valor varia entre -1 e 1, onde valores próximos a 1 indicam que o ponto está bem posicionado no seu _cluster_ e longe dos outros. Valores próximos a 0 indicam que o ponto está próximo da borda entre dois _clusters_. Funciona bem para _clusters_ compactos, como separados com k-médias. Considera tanto separação quanto coesão dos _clusters_.
    $$ s_i = \frac{b_i-a_i}{\max(a_i,b_i)} $$
    onde:
    - $a$ é a distância média entre o ponto e os pontos do seu próprio _cluster_;
    - $b$ é a menor distância média entre o ponto e todos os outros _clusters_;

In [205]:
def silhouette(Xy):
    n_clusters = np.unique(Xy[:,-1]).shape[0]
    si = []
    for sample_i in Xy:
        a = 0
        b_vec = []
        for cluster in range(n_clusters):
            cluster_samples = np.where(Xy[:,-1] == cluster)
            if cluster == sample_i[-1]:
                a += np.mean([np.linalg.norm(sample_i[:-1] - Xy[cluster_samples[0]][j][:-1]) for j in range(len(cluster_samples[0])) if not np.array_equal(Xy[cluster_samples[0]][j][:-1], sample_i[:-1]) ])
            else:
                b_dist = [np.linalg.norm(sample_i[:-1] - Xy[cluster_samples[0]][j][:-1]) for j in range(len(cluster_samples[0]))]
                b_vec.append(np.mean(b_dist))
        b = np.min(b_vec)
        si.append((b-a)/np.maximum(a,b))
    return np.mean(si)

print(f'Silhouette Score implementada manualmente: {silhouette(Xy):.4f}')
print(f'Silhouette Score usando sklearn: {silhouette_score(Xy[:,:-1], Xy[:,-1]):.4f}')


Silhouette Score implementada manualmente: 0.7415
Silhouette Score usando sklearn: 0.7415


- **Calinski-Harabasz Index ('Critério da Razão de Variância')**: maior pontuação significa que os _clusters_ são densos e bem separados. Entretanto é mais alto para _clusters_ convexos do que para outros conceitos de _clusters_ como os baseados em densidade. Bom para comparar diferentes métodos de clusterização e/ou diferentes parâmetros.
    $$ \begin{align} s &= \frac{tr(B_k)}{tr(W_k)} \times \frac{n-k}{k-1} \\ W_k&=\sum^k_{q=1} \sum _{x \in C_q} (x-c_q)(x-c_q)^T \\ B_k &= \sum_{q=1}^k n_q(c_q-c_E)(c_q-c_E)^T \end{align} $$
    onde:
    - $tr(x)$ é o traço de x;
    - $B_k$ é a disperção entre grupos;
    - $W_k$ é a disperção entre _clusters_;
    - $C_q$ são os pontos do _cluster_ $q$;
    - $c_q$ é o centro do _cluster_ $q$;
    - $c_E$ é o centro de E;
    - $n_q$ é o número de pontos no _cluster_ $q$
    - $k$ é o número de _clusters_;
    - $n_E$ é o número total de pontos

In [206]:
def calinski_harabasz(Xy):
    labels = Xy[:, -1]
    k = len(np.unique(labels))
    n = Xy.shape[0]

    global_center = np.mean(Xy[:, :-1], axis=0)
    Wk = 0.0
    Bk = 0.0 
    
    for cluster in np.unique(labels):
        cluster_points = Xy[labels == cluster, :-1]
        cluster_size = len(cluster_points)
        cluster_center = np.mean(cluster_points, axis=0)
        Wk += np.sum((cluster_points - cluster_center) ** 2)
        Bk += cluster_size * np.sum((cluster_center - global_center) ** 2)
    
    return (Bk / Wk) * ((n - k) / (k - 1))

print(f'Calinski-Harabasz Score implementada manualmente: {calinski_harabasz(Xy):.4f}')
print(f'Calinski-Harabasz Score usando sklearn: {calinski_harabasz_score(Xy[:,:-1], Xy[:,-1]):.4f}')

Calinski-Harabasz Score implementada manualmente: 1161.4263
Calinski-Harabasz Score usando sklearn: 1161.4263


- **Davies-Bouldin Index**: mede a separação entre os _clusters_ e a sua compactação através de um cálculo de 'similaridade'. Quanto menor o índice, melhor a separação entre os _clusters_. Penaliza _clusters_ que se sobrepõem ou que são muito esparsos. 
    $$ DB = \frac{1}{k} \sum_{i=1}^k \max_{j \neq i} \left( \frac{s_i + s_j}{d(c_i, c_j)} \right) $$
    onde:
    - $s_i$ é a dispersão do _cluster_ $i$ (distância média entre os pontos do _cluster_ e o centro do _cluster_);
    - $d(c_i, c_j)$ é a distância entre os centros dos _clusters_ $i$ e $j$;
    - $k$ é o número de _clusters_.

In [207]:
def davies_bouldin(Xy):
    k = np.unique(Xy[:,-1]).shape[0]
    c_q = np.zeros((k, Xy.shape[1] - 1))
    s = np.zeros(k)
    db = np.zeros(k)
    for cluster in range(k):
        cluster_samples = np.where(Xy[:,-1] == cluster)
        c_q[cluster] = np.mean(Xy[cluster_samples[0],:-1], axis=0)
        s[cluster] = np.mean([np.linalg.norm(Xy[idx,:-1] - c_q[cluster]) for idx in cluster_samples[0]])
    for cluster_i in range(k):
        db_aux = []
        for cluster_j in range(k):
            if cluster_i != cluster_j:
                db_aux.append((s[cluster_i] + s[cluster_j])/np.linalg.norm(c_q[cluster_i]-c_q[cluster_j]))
        db[cluster_i] = max(db_aux)
    return np.mean(db)

print(f'Davies-Bouldin Score implementada manualmente: {davies_bouldin(Xy):.4f}')
print(f'Davies-Bouldin Score usando sklearn: {davies_bouldin_score(Xy[:,:-1], Xy[:,-1]):.4f}')

Davies-Bouldin Score implementada manualmente: 0.3650
Davies-Bouldin Score usando sklearn: 0.3650


- **Dunn Index**: mede a separação entre os _clusters_ e a sua compactação através de um cálculo de 'similaridade'. Quanto maior o índice, melhor a separação entre os _clusters_. Mede diretamente a separação e compactação dos _clusters_.
    $$ D = \frac{min \; \delta(C_i,C_j)}{max \; \Delta(C_l)} $$
    onde:
    - $\delta(C_i,C_j)$ distância entre _clusters_ (separação);
    - $\Delta(C_l)$ distância dentro de cada _cluster_ (coesão).

In [208]:
def dunn(Xy):
    labels = Xy[:, -1]
    features = Xy[:, :-1]
    unique_labels = np.unique(labels)
    k = len(unique_labels)
    
    cluster_centers = np.array([np.mean(features[labels == label], axis=0) for label in unique_labels])
    
    s = np.zeros(k)
    for i, label in enumerate(unique_labels):
        cluster_points = features[labels == label]
        distances = np.linalg.norm(cluster_points - cluster_centers[i], axis=1)
        s[i] = np.max(distances)
    
    inter_distances = np.linalg.norm(cluster_centers[:, np.newaxis] - cluster_centers, axis=2)
    np.fill_diagonal(inter_distances, np.inf)
    min_inter_distances = np.min(inter_distances, axis=1)
    
    return np.min(min_inter_distances) / np.max(s)

print(f'Dunn Score implementado manualmente: {dunn(Xy):.4f}')

Dunn Score implementado manualmente: 1.9338


#### Testando métricas no Fuzzy C-means

In [212]:
# Definir os números de clusters a serem testados
cluster_range = range(2, 30)  # Testar de 2 a 10 clusters
fuzz_par = 1.1  # Parâmetro de fuzzificação

# Armazenar os resultados
results = []

for n_clusters in cluster_range:
    # Armazenar as métricas para cada iteração
    silhouette_scores = []
    calinski_harabasz_scores = []
    davies_bouldin_scores = []
    dunn_scores = []
    
    for _ in range(10):  # Rodar o FCM 10 vezes para cada número de clusters
        # Treinar o modelo FCM
        fcm_model = fuzz.cluster.cmeans(word_vectors.T, c=n_clusters, m=fuzz_par, error=0.05, maxiter=1000, init=None)
        cluster_membership = np.argmax(fcm_model[1], axis=0)
        
        # Adicionar os rótulos aos dados
        Xy = np.hstack((word_vectors, cluster_membership.reshape(-1, 1)))
        
        # Calcular as métricas
        silhouette_scores.append(silhouette(Xy))
        calinski_harabasz_scores.append(calinski_harabasz(Xy))
        davies_bouldin_scores.append(davies_bouldin(Xy))
        dunn_scores.append(dunn(Xy))
    
    # Calcular a média das métricas
    results.append({
        'n_clusters': n_clusters,
        'silhouette': np.mean(silhouette_scores),
        'calinski_harabasz': np.mean(calinski_harabasz_scores),
        'davies_bouldin': np.mean(davies_bouldin_scores),
        'dunn': np.mean(dunn_scores)
    })

# Converter os resultados em um DataFrame para análise
df_results = pd.DataFrame(results)

# Exibir os resultados
display(df_results)

# Plotar as métricas para análise com cada linha em uma cor diferente e adicionar uma legenda
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Silhouette Score", "Calinski-Harabasz Index", "Davies-Bouldin Index", "Dunn Index")
)

# Adicionar cada métrica em um subplot separado
fig.add_trace(
    px.line(df_results, x='n_clusters', y='silhouette', title='Silhouette Score', color_discrete_sequence=['blue']).data[0],
    row=1, col=1
)
fig.add_trace(
    px.line(df_results, x='n_clusters', y='calinski_harabasz', title='Calinski-Harabasz Index', color_discrete_sequence=['green']).data[0],
    row=1, col=2
)
fig.add_trace(
    px.line(df_results, x='n_clusters', y='davies_bouldin', title='Davies-Bouldin Index', color_discrete_sequence=['red']).data[0],
    row=2, col=1
)
fig.add_trace(
    px.line(df_results, x='n_clusters', y='dunn', title='Dunn Index', color_discrete_sequence=['purple']).data[0],
    row=2, col=2
)

# Atualizar layout para incluir títulos e ajustar o tamanho
fig.update_layout(
    title="Grid Search Metrics for FCM",
    width=1000,
    height=800
)

fig.show()

,n_clusters,silhouette,calinski_harabasz,davies_bouldin,dunn
0,2,0.057054,22.319934,3.869103,0.357349
1,3,0.048993,18.570247,3.620271,0.374506
2,4,0.050914,17.284566,3.319106,0.413689
3,5,0.058284,16.497098,3.099184,0.433581
4,6,0.062569,15.846266,2.944600,0.446543
5,7,0.068739,15.184542,2.796916,0.470591
6,8,0.073513,14.628262,2.688195,0.482650
7,9,0.074721,14.053390,2.614431,0.487231
8,10,0.075069,13.489319,2.570185,0.478501
9,11,0.075517,12.988910,2.544353,0.488095


#### External Clustering Indices
- Avalia a qualidade de uma clusterização utilizando rótulos/infomrações externas;
- Requer conhecimento prévio dos rótulos dos dados;
- Exemplos:
    - **Rand Index**: mede a similaridade entre dois _clusters_ (ou partições) comparando os pares de pontos. O valor varia entre 0 e 1, onde 1 indica que os dois _clusters_ são idênticos e 0 indica que não há similaridade.
        $$ RI = \frac{TP + TN}{TP + TN + FP + FN} $$
        onde:
        - $TP$ é o número de pares de pontos que estão no mesmo _cluster_ em ambas as partições;
        - $TN$ é o número de pares de pontos que estão em _clusters_ diferentes em ambas as partições;
        - $FP$ é o número de pares de pontos que estão no mesmo _cluster_ na primeira partição, mas em _clusters_ diferentes na segunda;
        - $FN$ é o número de pares de pontos que estão em _clusters_ diferentes na primeira partição, mas no mesmo _cluster_ na segunda.
    - **Adjusted Rand Index**: versão ajustada do Rand Index que leva em conta a chance de coincidência aleatória. O valor varia entre -1 e 1, onde 1 indica que os dois _clusters_ são idênticos, 0 indica que a similaridade é igual à chance aleatória e -1 indica que os dois _clusters_ são completamente diferentes.
        $$ ARI = \frac{RI - E[RI]}{\max(RI) - E[RI]} $$
        onde:
        - $E[RI]$ é a expectativa do Rand Index sob a hipótese nula de que as duas partições são independentes.
    - **Fowlkes-Mallows Index**: mede a similaridade entre dois _clusters_ (ou partições) comparando os pares de pontos. O valor varia entre 0 e 1, onde 1 indica que os dois _clusters_ são idênticos e 0 indica que não há similaridade.
        $$ FM = \frac{TP}{\sqrt{(TP + FP)(TP + FN)}} $$